# Chapter 4: From Straight Lines to Log-Odds

In Chapter 3 we predicted a number (sales). Now we want to predict a **category**: will a credit card customer default on their debt, yes or no?

- **Part 1** tries the tool we already have, linear regression, and finds where it breaks.
- **Part 2** finds a scale where a straight line *does* make sense. That scale has a natural meaning: it measures evidence. Logistic regression falls out of it.

### By the end, you'll be able to

1. Convert between probability and odds in both directions, and explain why odds range over $[0, \infty)$.
2. Use a likelihood ratio to update odds when a new clue arrives, and explain why evidence *multiplies* odds.
3. Explain why the same clue moves probability by different amounts depending on the starting point, but always moves log-odds by the same amount.
4. Explain why a linear model in log-odds means "each feature contributes its own additive amount of evidence."
5. Derive the logistic function from $\text{log-odds} = \beta_0 + \beta_1 X$ using algebra only.
6. Interpret $\beta_1$ as a multiplier on the odds, and avoid reading it as a change in probability.

> **The main arc:** Evidence multiplies odds, logs turn multiplying into adding, and a linear model is an adding machine. Logistic regression is what you get when you put those three facts together.

# Part 1: Can linear regression classify?

## Problem setup: Classification

#### *Your Boss:*
- "Which of our customers are going to default next year?"
- "How likely is it for *this* customer?"

We'll use ISL's `Default` data: 10,000 simulated credit card customers. For each one we know their monthly credit card `balance`, their annual `income`, whether they're a `student`, and whether they `default`ed.

In [ ]:
from ch_4_demos_helpers import load_default

# ISLP's Default data, plus a 0/1 copy of the default column: Yes -> 1, No -> 0
Default = load_default()
Default.head()

How common is defaulting?

In [ ]:
Default['default'].value_counts()

Only 333 of 10,000 customers (3.3%) defaulted. Defaulting is rare, which will matter later.

### A first look

This recreates ISL Figure 4.1. On the left, every defaulter is shown, but only 1,000 of the 9,667 non-defaulters, so the defaulters aren't buried. On the right, boxplots compare `balance` and `income` between the two groups.

In [ ]:
from ch_4_demos_helpers import plot_default_overview

plot_default_overview(Default);

**Discuss:** Which variable separates defaulters from non-defaulters better, `balance` or `income`?

<details>
<summary>What the plots show</summary>

`balance`. Defaulters mostly sit on the right of the scatterplot, and their balance boxplot barely overlaps the non-defaulters'. The two income boxplots look almost the same.

</details>

Like ISL, we'll use just one predictor from here on: `balance`.

## Try what we know: a straight line

Linear regression needs a numeric response, so code the labels as numbers:

$$Y = \begin{cases} 1 & \text{if the customer defaulted} \\ 0 & \text{if not} \end{cases}$$

Then fit a least squares line, exactly as in Chapter 3:

$$\hat{Y} = \hat\beta_0 + \hat\beta_1 \cdot \texttt{balance}$$

This isn't as silly as it sounds. The average of a column of 0s and 1s is the *fraction* of 1s. So when least squares tries to predict the average $Y$ at each balance, it's really trying to estimate the **probability of default** at that balance. If this works, $\hat{Y}$ is a probability.

In [ ]:
from sklearn.linear_model import LinearRegression

line = LinearRegression().fit(Default[['balance']], Default['default01'])
print(f'intercept = {line.intercept_:.4f}, slope = {line.coef_[0]:.6f} per dollar')

Here's the line over the data. Every customer is a dot at 0 (didn't default) or 1 (defaulted), with a little vertical jitter so the dots don't all stack on top of each other. The gray bands are values no probability can take.

In [ ]:
from ch_4_demos_helpers import plot_linear_fit

plot_linear_fit(Default, line);

### Problem 1: "probabilities" below zero

What does the line predict for a few balances?

In [ ]:
from ch_4_demos_helpers import linear_prediction_table

linear_prediction_table(line)

**Discuss:** A customer with a \$0 balance gets a predicted value of about $-0.075$. What would a probability of $-7.5\%$ mean?

<details>
<summary>Answer</summary>

Nothing. A probability can't be negative. Yet the line gives a negative value to every customer with a balance below about \$580, which is 31% of the customers in the data.

</details>

The line would also go above 1 if we followed it far enough to the right. Any straight line with a nonzero slope eventually leaves $[0, 1]$ in at least one direction.

### Problem 2: the line can't bend

To see what the line *should* look like, group customers into \$200-wide balance bins and compute the fraction in each bin who actually defaulted. Each red dot below is one bin; bigger dots hold more customers.

In [ ]:
from ch_4_demos_helpers import binned_default_rate, plot_binned_vs_line

plot_binned_vs_line(Default, line);

**Discuss:** Describe the shape the red dots make. Where is the line too high? Too low?

<details>
<summary>What the plot shows</summary>

The default rate stays near 0 until a balance of about \$1,200, then climbs steeply and levels off near 1: an S shape. A straight line can't bend, so it compromises everywhere. It's below zero at small balances, slightly too high in the middle, and far too low at large balances. In the \$2,000–\$2,200 bin, 73% of customers defaulted, but the line predicts about 19%.

</details>

### Problem 3: as a classifier, it flags no one

The natural rule for turning a probability into a yes/no prediction is: predict "default" when the probability is above 0.5. How does the line do with that rule?

The line's highest prediction is about 0.27, at the largest balance in the data, so it never crosses 0.5. It predicts "no default" for all 10,000 customers, including the 106 customers with balances over \$2,000, 81 of whom actually defaulted.

**Discuss:** We could lower the cutoff until the line flags someone. Why doesn't that fix the underlying problem?

<details>
<summary>One answer</summary>

A different cutoff might give reasonable yes/no predictions, but the line's outputs still aren't trustworthy probabilities. Your boss's second question, "How likely is it for *this* customer?", still gets answers like $-7.5\%$.

</details>

### Problem 4: more than two categories

ISL (§4.2) gives one more reason. Suppose we're predicting the diagnosis of a patient in the emergency room, with three possibilities. Coding them as numbers

$$Y = \begin{cases} 1 & \text{stroke} \\ 2 & \text{drug overdose} \\ 3 & \text{epileptic seizure} \end{cases}$$

tells linear regression that the outcomes are in order, and that the gap between stroke and overdose equals the gap between overdose and seizure. Nothing about the medicine says that. A different, equally reasonable coding gives a different model and different predictions. With two categories the 0/1 coding is harmless (flipping it just flips the line), but the problems above remain.

## What do we need instead?

**Discuss:** What do we need from a model whose output is a probability?

<details>
<summary>Answer</summary>

Its output must always land between 0 and 1, no matter what $X$ is. Ideally it should also be able to bend, staying flat where the data is flat and rising steeply where the data rises steeply.

</details>

Here's a preview of where we're headed: the same data with the linear regression line (left), and the *logistic regression* curve we'll build in Part 2 (right). This is ISL Figure 4.2.

In [ ]:
from ch_4_demos_helpers import plot_linear_vs_logistic

plot_linear_vs_logistic(Default, line);

The curve on the right stays between 0 and 1 and bends to follow the data. But where does an S-curve come from? We could just say "pick an S-shaped function," but there are many of those. In Part 2 we'll find a scale where a straight line *does* make sense, and the S-curve will come out of the algebra.

# Part 2: Choosing the model: evidence adds

A straight line can't stay between 0 and 1 by itself. Instead of bending the line, we'll change the *scale* we measure probability on, until we find one where a straight line makes sense. It turns out that scale has a natural meaning: it measures evidence.

## Odds

**Odds compare "yes" to "no."** If 1 customer in 10 defaults, then for every 1 defaulter there are 9 non-defaulters. The odds of default are **1 to 9**, written $1 : 9$ or $1/9 \approx 0.111$.

In general, if $p$ is the probability of "yes":

$$\text{odds} = \frac{p}{1-p} \qquad\qquad p = \frac{\text{odds}}{1 + \text{odds}}$$

### Practice

Fill in the blanks. Some rows start from the probability, some from the odds.

| Probability | Odds (ratio) | Odds (number) |
|---|---|---|
| 10% | 1 : 9 | 0.111 |
| 20% | ? | ? |
| 50% | ? | ? |
| ? | 3 : 1 | ? |
| ? | ? | 4 |
| 99% | ? | ? |
| ? | 1 : 99 | ? |

<details>
<summary>Answers</summary>

| Probability | Odds (ratio) | Odds (number) |
|---|---|---|
| 10% | 1 : 9 | 0.111 |
| 20% | 1 : 4 | 0.25 |
| 50% | 1 : 1 | 1 |
| 75% | 3 : 1 | 3 |
| 80% | 4 : 1 | 4 |
| 99% | 99 : 1 | 99 |
| 1% | 1 : 99 | 0.0101 |

</details>

### What range can odds take?

**Discuss:** What happens to the odds as $p$ gets close to 1? Close to 0?

<details>
<summary>Answer</summary>

Near 1, the odds explode toward infinity: 99% is 99 : 1, and 99.9% is 999 : 1. Near 0, the odds shrink toward 0 but never go negative.

</details>

$$p \in [0, 1] \quad\longrightarrow\quad \text{odds} \in [0, \infty)$$

Keep this in mind. We're partway to a scale with no limits at all, which is what a straight line needs.

## Demo: Probability vs. odds

Odds plotted against probability, with the practice-table rows marked in gray. Use the slider to move along the curve; the readout gives the odds as a ratio and as a number. The odds stay small for most of the range, then shoot upward as $p$ approaches 100%.

In [ ]:
from ch_4_demos_helpers import odds_explorer

odds_explorer()

## Evidence multiplies odds

This is the heart of Part 2. We'll work with **counts of people** rather than probability formulas; the key rule falls right out of the counting.

*The numbers in this section are invented for the illustration. They aren't from ISL's `Default` data.*

### The setup

A bank has **1,000 customers**. Over the next year, **100 will default** and **900 won't**.

$$\text{Starting odds of default} = 100 : 900 = 1 : 9 \qquad (p = 10\%)$$

Before we know anything about a particular customer, our best guess is $1 : 9$ odds of default. Now we'll learn things about them, one clue at a time.

### Clue 1: the customer missed a payment last month

- **60%** of defaulters missed a payment last month (so 40% of them didn't).
- **20%** of non-defaulters missed a payment last month (so 80% of them didn't).

Split each group by the clue:

$$\underbrace{100 \text{ defaulters}}_{\substack{60 \text{ missed} \\ 40 \text{ didn't}}} \qquad\qquad \underbrace{900 \text{ non-defaulters}}_{\substack{180 \text{ missed} \\ 720 \text{ didn't}}}$$

Our customer missed a payment (clue 1), so they're in one of the two "missed" groups. **Ignore everyone who didn't miss a payment** and count what's left:

$$60 \text{ defaulters} : 180 \text{ non-defaulters} \;=\; 1 : 3 \qquad (p = 25\%)$$

## Demo: Splitting 1,000 customers

The same split as a tree. Both groups split by the clue; we follow the branch that matches our customer (colored) and drop the other (gray). The odds of the customers still in the running are in the middle.

In [ ]:
from ch_4_demos_helpers import plot_customer_tree

plot_customer_tree(n_clues=1);

**Discuss:** The odds went from $1 : 9$ to $1 : 3$. What did they get multiplied by? Where did that number come from?

<details>
<summary>Answer</summary>

They were multiplied by 3, which is the ratio of the two rates from the clue:

$$\frac{\text{rate of missed payments among defaulters}}{\text{rate of missed payments among non-defaulters}} = \frac{60\%}{20\%} = 3$$

</details>

We'll call this number the **strength of the clue**. (Its formal name is the **likelihood ratio**.) It answers: *how many times more common is this clue among defaulters than among non-defaulters?*

$$\boxed{\text{new odds} = \text{old odds} \times \text{strength of the clue}}$$

$$\frac{1}{9} \times 3 = \frac{3}{9} = \frac{1}{3} \;\checkmark$$

We didn't need to split up the customers. We could have just multiplied by 3. The counting is there to convince you the shortcut is correct.

### Why does the shortcut work?

Every defaulter is kept with chance 60%, and every non-defaulter is kept with chance 20%. So the "yes" side of the odds gets multiplied by 0.6 and the "no" side by 0.2, and the ratio between them gets multiplied by $0.6 / 0.2 = 3$. The total number of customers never enters into it.

### Clue 2: the customer has a high balance

- **50%** of defaulters have a high balance.
- **25%** of non-defaulters have a high balance.

**Discuss:** What's the strength of this clue? Predict the new odds before we count.

<details>
<summary>Answer</summary>

$$\text{strength} = \frac{50\%}{25\%} = 2 \qquad\qquad \text{new odds} = \frac{1}{3} \times 2 = \frac{2}{3}$$

</details>

Now check by counting. Start from the customers still in the running (the 60 and the 180) and keep only those with a high balance:

$$60 \times 50\% = 30 \text{ defaulters} \qquad\qquad 180 \times 25\% = 45 \text{ non-defaulters}$$

$$30 : 45 = 2 : 3 \qquad (p = 40\%) \;\checkmark$$

> **An important assumption.** We assumed that among defaulters, having a high balance is unrelated to missing a payment, and the same among non-defaulters. That's why we could apply the 50% and 25% to the groups that remained. Real clues often overlap. We'll come back to this.

### Clue 3: the customer has been with the bank for over 10 years

- **20%** of defaulters are long-time customers.
- **40%** of non-defaulters are long-time customers.

**Discuss:** Strength? New odds?

<details>
<summary>Answer</summary>

$$\text{strength} = \frac{20\%}{40\%} = 0.5 \qquad\qquad \text{new odds} = \frac{2}{3} \times 0.5 = \frac{1}{3} \quad (\text{back to } 25\%)$$

Check with counts: $30 \times 20\% = 6$ defaulters and $45 \times 40\% = 18$ non-defaulters, so $6 : 18 = 1 : 3$. ✓

</details>

A clue with strength less than 1 is evidence *against* default. A clue with strength exactly 1 is equally common in both groups, so it tells us nothing and leaves the odds unchanged.

In [ ]:
# The full tree, through all three clues
plot_customer_tree(n_clues=3);

### Summary

| Step | Clue | Strength | Odds | Probability |
|---|---|---|---|---|
| Start | (nothing yet) | — | 1 : 9 | 10% |
| 1 | missed a payment | × 3 | 1 : 3 | 25% |
| 2 | high balance | × 2 | 2 : 3 | 40% |
| 3 | long-time customer | × 0.5 | 1 : 3 | 25% |

**Discuss:** Would we get a different answer if we learned the clues in a different order?

<details>
<summary>Answer</summary>

No. Multiplication is "commutative" (order doesn't change the result):

$$\frac{1}{9} \times 3 \times 2 \times 0.5 \;=\; \frac{1}{9} \times 0.5 \times 2 \times 3$$

</details>

## Conclusion

$$\text{new odds} = \text{old odds} \times \text{strength}$$    



## Probability Problem: Same clue, different jumps in probability

Probability turns out to be the wrong scale for describing what a clue does. Apply clue 1, the missed payment with strength 3, to four customers who start at different risk levels:

| Starting probability | Starting odds | Odds × 3 | New probability | Change in probability |
|---|---|---|---|---|
| 1% | 1 : 99 | 3 : 99 = 1 : 33 | 2.9% | +1.9 points |
| 10% | 1 : 9 | 3 : 9 = 1 : 3 | 25% | +15 points |
| 50% | 1 : 1 | 3 : 1 | 75% | +25 points |
| 90% | 9 : 1 | 27 : 1 | 96.4% | +6.4 points |

Try one row yourself before reading it off the table.

**Discuss:** If someone asked "how much does a missed payment increase the probability of default?", what's the honest answer?

<details>
<summary>Answer</summary>

It depends on where you start. There's no single number.

</details>

**Discuss:** Is there a way to describe the clue's effect with one number?

<details>
<summary>Answer</summary>

Yes: it always multiplies the odds by 3.

</details>

Multiplying is nice, but our linear models *add*. We need a scale where this clue always *adds* the same amount.

## Log-odds Scale: Effect of Evidence

when we apply the effect of evidence (evidence strength), for some particular evidence:

- Probabilities change by amounts that depend upon the starting probability, when new evidence is included.
- log(odds) always changes by the same amount regardless of the initial log(odds)

The interactive widget below demonstrates these facts.

In [ ]:
from ch_4_demos_helpers import one_clue_explorer

one_clue_explorer(start=3)

<details>
<summary>What to look for</summary>

At strength 1, the arrows vanish: the clue is equally common in both groups, so it tells us nothing. Below 1, the arrows point down, so the clue is evidence against default. At every strength, the log-odds arrows all have the same length, $\ln(\text{strength})$.

</details>

## Logs turn multiplying into adding

### Three facts about logs

We'll use the natural log, $\ln$, to match ISL. You only need three facts, each with a quick check:

| Fact | Check |
|---|---|
| $\ln(a \times b) = \ln a + \ln b$ | $\ln 6 = 1.79 = 0.69 + 1.10 = \ln 2 + \ln 3$ |
| $\ln 1 = 0$ | multiplying by 1 changes nothing, and adding 0 changes nothing |
| $\ln\!\left(\tfrac{1}{a}\right) = -\ln a$ | $\ln 0.5 = -0.69 = -\ln 2$ |

### The evidence tally in log-odds

Suppose we update some *old odds* using the *strength* of a clue.
We have something like:


$$\text{new odds} = \text{old odds} \times \text{strength}$$    

Take the log of both sides of the rule:

$$\ln(\text{new odds}) = \ln(\text{old odds}) + \ln(\text{strength})$$

The log of the odds is called the **log-odds** (or **logit**). Here is the clue table again, on the log-odds scale:

| Step | Clue | $\ln(\text{strength})$ | Log-odds | Probability |
|---|---|---|---|---|
| Start | — | — | $\ln(1/9) = -2.20$ | 10% |
| 1 | missed a payment | $\ln 3 = $ **+1.10** | $-1.10$ | 25% |
| 2 | high balance | $\ln 2 = $ **+0.69** | $-0.41$ | 40% |
| 3 | long-time customer | $\ln 0.5 = $ **−0.69** | $-1.10$ | 25% |

Log-odds is a **running score of evidence**. You start with a baseline score, and every clue adds or subtracts its own fixed number of points. Clues for default add points, clues against default subtract points, and useless clues add zero.


### Three scales for the same thing

| Probability | Odds | Log-odds |
|---|---|---|
| 1% | 0.0101 | −4.60 |
| 10% | 0.111 | −2.20 |
| 50% | 1 | 0 |
| 90% | 9 | +2.20 |
| 99% | 99 | +4.60 |

Notice the symmetry: 10% and 90% have log-odds that are exact opposites, and so do 1% and 99%. Even odds (50%) sit exactly at 0.

$$p \in [0,1] \;\longrightarrow\; \text{odds} \in [0,\infty) \;\longrightarrow\; \text{log-odds} \in (-\infty, \infty)$$

Log-odds can be any number at all, positive or negative. That's exactly the range a straight line produces. **This is the scale where a linear model is allowed to live.**

## Demo: Three number lines

The five rows of the table, plotted on all three scales, with matching points connected. Notice:

- On the probability line, 90% and 99% are close together, but on the log-odds line they're about as far apart as 50% and 90%.
- On the odds line, everything at or below 50% is crammed between 0 and 1, while 99% is way out at 99.
- On the log-odds line, the points are symmetric around 0.

In [ ]:
from ch_4_demos_helpers import plot_three_scales

plot_three_scales();

## From clues to a model

### From yes/no clues to numeric features

So far, each clue was yes or no. But `balance` is a number. The natural extension: **every additional dollar of balance adds the same small amount of evidence.**

$$\underbrace{\ln\!\left(\frac{p(X)}{1-p(X)}\right)}_{\text{evidence score}} = \underbrace{\beta_0}_{\text{baseline}} + \underbrace{\beta_1 X}_{\text{evidence from balance}}$$

- $\beta_0$ is the starting score: the log-odds for a customer with $X = 0$. (If balance is standardized, $X = 0$ is an average customer, which is a much more meaningful baseline.)
- $\beta_1$ is the evidence per unit of $X$. A positive $\beta_1$ means a higher balance points toward default.

With more features, each one contributes its own term, exactly like adding clues:

$$\ln\!\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots$$

A linear model is an adding machine, and log-odds is the scale where evidence adds. So "linear in log-odds" says: **each feature contributes its own amount of evidence, and we add it all up.**

### Solving for $p$ (algebra only)

The model gives us log-odds, but we want a probability. To keep things tidy, write $z$ for the evidence score $\beta_0 + \beta_1 X$, then solve for $p$:

$$
\begin{aligned}
\ln\!\left(\frac{p}{1-p}\right) &= z \\[4pt]
\frac{p}{1-p} &= e^{z} && \text{undo the log: exponentiate both sides} \\[4pt]
p &= e^{z}(1-p) = e^{z} - p\,e^{z} && \text{multiply both sides by } (1-p) \\[4pt]
p + p\,e^{z} &= e^{z} && \text{move every } p \text{ term to the left} \\[4pt]
p\,(1 + e^{z}) &= e^{z} && \text{factor out } p \\[4pt]
p &= \frac{e^{z}}{1 + e^{z}} && \text{divide}
\end{aligned}
$$

Putting $\beta_0 + \beta_1 X$ back in for $z$:

$$\boxed{\,p(X) = \frac{e^{\beta_0 + \beta_1 X}}{1 + e^{\beta_0 + \beta_1 X}}\,}$$

This is ISL's equation (4.2), the **logistic function**. We never chose an S-curve. We chose "evidence adds," and the S-curve came out of the algebra.

Dividing the top and bottom by $e^{z}$ gives an equivalent form you'll see in scikit-learn's documentation and in neural network material:

$$p = \frac{1}{1 + e^{-z}}$$

### Sanity checks

| Evidence score $z$ | $e^{z}$ | $p$ | In words |
|---|---|---|---|
| very negative | close to 0 | close to 0 | overwhelming evidence against |
| 0 | 1 | 1/2 | evidence perfectly balanced |
| very positive | huge | close to 1 | overwhelming evidence for |

The curve flattening out at 0 and 1 isn't a feature we bolted on. It's what "a lot of evidence" means.

## Demo: One model, two views

Each figure below shows the same model two ways, using the `Default` data with `balance` standardized (0 is an average balance, and 1 is one standard deviation, about \$484, above average):

- **Left, the evidence view:** the straight line $z = \beta_0 + \beta_1 x$.
- **Right, the probability view:** the S-curve $p(x)$ over the data.

The filled dot marks a customer at $x$, and the open dot marks a customer one unit to the right. The readout at the top gives their log-odds, odds, and probability. Each figure has one slider; everything else stays fixed. The fixed values start at the best-fitting $\beta_0$ and $\beta_1$.

### Slide $\beta_0$

**Try it:** Increase $\beta_0$ only. What changes in each panel?

In [ ]:
from ch_4_demos_helpers import two_views_explorer

two_views_explorer(Default, vary='b0')

<details>
<summary>What to look for</summary>

The line slides up and the curve slides left. Baseline evidence goes up for everyone, so every customer reaches any given probability at a lower balance.

</details>

### Slide $\beta_1$

**Try it:**

1. Increase $\beta_1$. What changes?
2. Set $\beta_1 = 0$. What do both panels look like?
3. Make $\beta_1$ negative. What would that say about balance and default?

In [ ]:
two_views_explorer(Default, vary='b1')

<details>
<summary>What to look for</summary>

1. The line tilts more steeply and the S-curve gets steeper: each unit of balance carries more evidence.
2. A flat line and a flat curve. Balance carries no evidence, so every customer gets the same probability.
3. Higher balances would be evidence *against* default. The line slopes down and the S-curve flips to go downhill.

</details>

### Move the marker

**Try it:** Slide the marker from left to right. Watch the readout: does one unit to the right always change the probability by the same amount? Does it always multiply the odds by the same amount?

In [ ]:
two_views_explorer(Default, vary='marker')

<details>
<summary>What to look for</summary>

The odds always get multiplied by the same $e^{\beta_1} \approx 14.3$. The probability changes by different amounts: barely at all on the far left, the most near 50%, and less again on the far right.

</details>


# Learning Check Prep

Wednesday's learning check covers the ideas below. Try each question on paper before you open its answer.

### Question 1: Probability to odds

I'm giving you probabilities in ratio form to make conversions easier. Normally, you'd see the probability $\tfrac{1}{5}$, written as 0.2.  Write each probability of default as odds of default in ratio form ($a : b$). 

(a) $\tfrac{1}{5}$ &emsp; (b) $\tfrac{1}{4}$ &emsp; (c) $\tfrac{2}{3}$ &emsp; (d) $\tfrac{3}{10}$ &emsp; (e) $\tfrac{9}{10}$

<details>
<summary>Answer</summary>

Odds compare "yes" to "no": out of the whole group, how many default versus how many don't.

(a) 1 in 5 default, so 4 in 5 don't: $1 : 4$

(b) $1 : 3$

(c) $2 : 1$

(d) $3 : 7$

(e) $9 : 1$

</details>

### Question 2: Odds to probability

Write each odds of default as a probability of default in ratio form (a fraction).

(a) $1 : 3$ &emsp; (b) $1 : 1$ &emsp; (c) $2 : 3$ &emsp; (d) $4 : 1$ &emsp; (e) $3 : 5$

<details>
<summary>Answer</summary>

Odds of $a : b$ means $a$ defaulters for every $b$ non-defaulters, so $a$ out of every $a + b$ customers default: $p = \tfrac{a}{a+b}$.

(a) $\tfrac{1}{4}$

(b) $\tfrac{1}{2}$

(c) $\tfrac{2}{5}$

(d) $\tfrac{4}{5}$

(e) $\tfrac{3}{8}$

</details>

### Question 3: Updating odds with a likelihood ratio

A customer starts with odds of default $1 : 6$ (a 1/7 probability).

(a) We learn the customer recently opened several new credit cards. This happens for **30%** of defaulters and **10%** of non-defaulters. What is the likelihood ratio (strength) of this clue? What are the customer's new odds and new probability of default?

(b) Next we learn the customer has a stable job. This is true of **20%** of defaulters and **40%** of non-defaulters. Update the odds from part (a). What is the new probability?

(c) A third clue is true of 25% of defaulters and 25% of non-defaulters. What does it do to the odds?

<details>
<summary>Answer</summary>

$$\text{new odds} = \text{old odds} \times \text{strength}, \qquad \text{strength} = \frac{\text{rate among defaulters}}{\text{rate among non-defaulters}}$$

(a) Strength $= 30\% / 10\% = 3$. New odds $= \tfrac{1}{6} \times 3 = \tfrac{3}{6} = 1 : 2$, so $p = \tfrac{1}{3} \approx 0.33$.

(b) Strength $= 20\% / 40\% = \tfrac{1}{2}$, which is evidence *against* default. New odds $= \tfrac{1}{2} \times \tfrac{1}{2} = 1 : 4$, so $p = \tfrac{1}{5} = 0.2$.

(c) Strength $= 25\% / 25\% = 1$. Multiplying by 1 leaves the odds unchanged at $1 : 4$. A clue that's equally common in both groups is not evidence for or against default.

</details>

### Question 4: Linear regression as a probability model

A bank codes $Y = 1$ for customers who defaulted and $Y = 0$ for those who didn't, fits a line $\hat Y = \hat\beta_0 + \hat\beta_1 \cdot \texttt{balance}$ (using least squares), and treats $\hat Y$ as the probability of default.

Identify at least three problems with this approach.

<details>
<summary>Answer</summary>

Any three of these:

1. **Impossible probabilities.** A straight line with a nonzero slope eventually leaves $[0, 1]$. On the `Default` data it predicts about $-7.5\%$ for a \$0 balance, and it goes negative for about 31% of customers. Far enough to the right, it would go above 1.
2. **It can't bend.** The actual default rate is S-shaped: flat near 0, then rising steeply, then leveling off near 1. A straight line compromises everywhere. It's too high in the middle and far too low at high balances (it predicts about 19% where 73% actually defaulted).
3. **It's a poor classifier.** Defaults are rare, so the line stays low. Its highest prediction is about 0.27, so with a 0.5 cutoff it predicts "no default" for every customer. Lowering the cutoff might fix the yes/no predictions, but the outputs still aren't trustworthy probabilities.
4. **More than two categories.** Coding three or more outcomes as 1, 2, 3 tells the model they're in order and equally spaced. A different, equally reasonable coding gives a different model.

</details>

### Question 5: Why model log-odds as a linear combination?

Logistic regression assumes
$$\ln\!\bigg(\underbrace{\frac{p}{1-p}}_{\text{odds}}\bigg) = \underbrace{\beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots}_{\text{linear combination}}$$

Give a few reasons log-odds is a good thing to model as a linear combination, when probability and odds are not.

<details>
<summary>Answer</summary>

Any of these:

1. **Range.** A linear combination can output any real number. Probability is stuck in $[0, 1]$ and odds in $[0, \infty)$, but log-odds ranges over $(-\infty, \infty)$, just like a line.
2. **Evidence adds on this scale.** Each piece of evidence *multiplies* the odds by its strength. Taking logs turns that multiplication into addition: $\ln(\text{new odds}) = \ln(\text{old odds}) + \ln(\text{strength})$. A linear model is an adding machine, and each term $\beta_j X_j$ can be thought of as one feature's contribution of evidence when we add them up.
3. **A fixed effect per unit.** On the log-odds scale, a clue (or one unit of $X_j$) shifts the score by the same amount no matter where you start, as we saw in the demo above. That's exactly what a coefficient $\beta_j$ in a linear model says.
4. **The S-curve comes for free.** Solving for $p$ gives $p = \dfrac{e^{z}}{1 + e^{z}}$, which always lands strictly between 0 and 1 and bends to match S-shaped data, fixing the problems from Question 4.
5. **Label symmetry.** Swapping which outcome we call "1" just flips the signs of the coefficients: $\text{log-odds(no)} = -\text{log-odds(yes)}$. A model linear in odds wouldn't stay linear under that swap.

</details>

## Optional: Why not linear in odds?

(Everything below here is optional/extra)

**Discuss:** Why didn't we just set the odds equal to $\beta_0 + \beta_1 X$? That's simpler.

<details>
<summary>Two reasons, and a third from earlier</summary>

1. **Range.** Odds can't be negative, but a straight line eventually goes negative. This is Part 1's problem again, on one side instead of two.
2. **The labels would matter.** The odds of "no default" are $1/(\text{odds of default})$. If the odds of default were a straight line, the odds of *no* default would be $1/(\text{a straight line})$, which isn't a straight line. The model would change depending on which outcome we happened to call "1."
3. **That isn't how evidence works.** Clues *multiply* the odds; they don't add to them.

</details>

Compare log-odds. Swapping the labels gives

$$\text{log-odds(no)} = -\,\text{log-odds(yes)} = -\beta_0 - \beta_1 X$$

which is still a straight line. Swapping the labels just flips the signs of the coefficients, which is how it should be.

## Demo: Linear in odds breaks both ways

Take one made-up line, $0.5 + 1.0\,X$, and try it both ways.

- **Top row:** suppose the *odds* of default are this line. On the left, the line goes negative, which odds can't do. On the right, the odds of *no* default that it implies are $1/(0.5 + X)$, which isn't a line at all: it blows up where the left line crosses 0.
- **Bottom row:** suppose the *log-odds* of default are this line instead. Then the log-odds of no default are just the same line flipped upside down.

In [ ]:
from ch_4_demos_helpers import plot_linear_in_odds

plot_linear_in_odds(b0=0.5, b1=1.0);

## Interpreting the coefficients

### The rule

Increasing $X$ by one unit adds $\beta_1$ to the log-odds, so it **multiplies the odds by $e^{\beta_1}$**.

### ISL's fitted `Default` model (Table 4.1)

$$\hat\beta_0 = -10.6513 \qquad\qquad \hat\beta_1 = 0.0055 \text{ per dollar}$$

One dollar is too small a step to talk about, so use \$100:

$$e^{0.0055 \times 100} = e^{0.55} \approx 1.73$$

**Every extra \$100 of balance multiplies the odds of default by about 1.73, regardless of the customer's starting balance.**

### Predictions (matching ISL §4.3.3)

| Balance | Log-odds | Odds | Probability |
|---|---|---|---|
| \$1,000 | $-10.65 + 5.5 = -5.15$ | 0.0058 | 0.58% |
| \$2,000 | $-10.65 + 11 = 0.35$ | 1.42 | 58.6% |

**Discuss:** Going from \$1,000 to \$2,000, the probability jumped by about 58 points. Does that mean every extra \$1,000 adds 58 points?

<details>
<summary>Answer</summary>

No. An extra \$1,000 always multiplies the odds by $e^{5.5} \approx 245$. What that does to the probability depends on where you start, exactly like the four customers earlier.

</details>

> **$\beta_1$ is not a change in probability.** It's a change in log-odds, which is a fixed multiplier on the odds. The change in probability depends on where you are on the curve: it's biggest near 50% and smallest near 0% and 100%.

## Demo: Same \$100, different jumps

First, fit the model ourselves with `statsmodels` and check that we get ISL's Table 4.1. (How the fitting works is Part 3.)

In [ ]:
from ch_4_demos_helpers import fit_logistic, logistic_table, next_100_table, plot_next_100

logit_fit = fit_logistic(Default)   # logistic regression of default01 on balance
logistic_table(logit_fit)

Now follow customers from \$1,000 to \$2,500 and ask what one more \$100 of balance does to each of them:

In [ ]:
next_100_table(logit_fit)

The odds multiplier column never changes: 1.733 every time. The change in probability ranges from under half a point to over 12 points. Here is the same comparison at every balance:

In [ ]:
plot_next_100(logit_fit);

**Discuss:** Where does an extra \$100 matter most? Why there?

<details>
<summary>Answer</summary>

Around \$1,940, where the fitted probability is 50%. That's where the S-curve is steepest. Far to the left, customers are almost certainly safe, and far to the right they're almost certainly defaulting, so the same multiplier on the odds barely moves the probability.

</details>

## Up next

We now have a model that always gives a probability between 0 and 1, and whose coefficients mean something: each one is an amount of evidence. But in the "One model, two views" demo we picked $\beta_0$ and $\beta_1$ by hand. **How do we decide which values are *best*?** In Chapter 3, least squares answered that question for lines. Part 3 answers it for logistic regression, using *maximum likelihood*.

## Check your understanding

1. A customer's probability of default is 20%. What are their odds?
2. A clue occurs in 30% of defaulters and 10% of non-defaulters. A customer with $1 : 4$ odds has this clue. What are their new odds?
3. True or false: in a logistic regression with $\beta_1 = 0.7$, increasing $X$ by one unit raises the probability by 0.7.
4. Why does a linear model belong on the log-odds scale rather than the probability scale?

<details>
<summary>Answers</summary>

1. $1 : 4$, or 0.25.
2. The strength is $30\% / 10\% = 3$, so the new odds are $1/4 \times 3 = 3 : 4$.
3. False. It raises the *log-odds* by 0.7, which multiplies the odds by $e^{0.7} \approx 2$. The change in probability depends on where you start.
4. Log-odds can be any real number, just like a line's output, and evidence adds on the log-odds scale.

</details>

## Going further (optional)

### Bayes' theorem in odds form

If you've seen Bayes' theorem, the counting argument for clues is exactly this. Write $D$ for "defaults," $N$ for "doesn't default," and $C$ for "has the clue":

$$\frac{P(D \mid C)}{P(N \mid C)} = \frac{P(C \mid D)\,P(D)}{P(C)} \Bigg/ \frac{P(C \mid N)\,P(N)}{P(C)} = \underbrace{\frac{P(D)}{P(N)}}_{\text{old odds}} \times \underbrace{\frac{P(C \mid D)}{P(C \mid N)}}_{\text{strength}}$$

The $P(C)$ terms cancel. That's the formal version of "the total number of customers never enters into it."

### When clues overlap

Back in clue 2, we assumed the clues were unrelated within each group. What goes wrong if clue 2 is "missed *two* payments" instead? It overlaps heavily with clue 1: almost everyone who missed two payments also missed one. Multiplying both strengths counts the same evidence twice and makes us overconfident.

Multiplying clue strengths that were each computed on their own is exactly what **naive Bayes** does (later in Chapter 4). Logistic regression is different: it fits all the $\beta$'s *together*, so overlapping features share the credit instead of being counted twice. That's a strong point in favor of logistic regression.

### Label symmetry

Show that $1 - p(X)$ is the logistic function of $-(\beta_0 + \beta_1 X)$.

<details>
<summary>Solution</summary>

With $z = \beta_0 + \beta_1 X$:

$$1 - \frac{e^{z}}{1 + e^{z}} = \frac{1 + e^{z} - e^{z}}{1 + e^{z}} = \frac{1}{1 + e^{z}} = \frac{e^{-z}}{e^{-z} + 1}$$

The last step multiplies the top and bottom by $e^{-z}$. The result is the logistic function evaluated at $-z$.

</details>

## Demo: Counting the same evidence twice

Here are 10,000 simulated customers. Missing *two* payments only happens to people who also missed *one*, so the second clue overlaps completely with the first. First, each clue's strength on its own:

In [ ]:
from ch_4_demos_helpers import simulate_overlapping_clues, clue_strengths, overlap_comparison

customers = simulate_overlapping_clues()
clue_strengths(customers)

Now three answers for the odds of default for a customer with *both* clues:

1. **Naive:** start at the overall odds and multiply by both strengths.
2. **Counting:** look at the customers who actually have both clues and count.
3. **Logistic regression** fit on both clues together.

In [ ]:
overlap_comparison(customers)

The naive answer says about 75%, but only about half of these customers actually defaulted. Multiplying both strengths counted the missed payment twice. Logistic regression fits both coefficients together, so the second clue only gets credit for what it adds beyond the first, and it matches the counted answer.

(It matches *exactly* here because there are only three kinds of customers: neither clue, one missed payment, or two. Three kinds of customers and three coefficients means the model can fit each group's rate perfectly. With messier data it won't be exact, but it still avoids the double counting.)